In [1]:
"""
Aliquot 18ul qPCR Mix into 96-well plate and 2ul Sample from Diluted Series

Author : Harley King
Date   : 2025-07-21
"""


'\nAliquot 18ul qPCR Mix into 96-well plate and 2ul Sample from Diluted Series\n\nAuthor : Harley King\nDate   : 2025-07-21\n'

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.opentrons.tube_racks import (
    opentrons_24_tuberack_generic_1point5ml_snapcap_short,
)
from pylabrobot.resources.tube_adapter import TubeRackAdapter
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources import (
    TIP_50ul_w_filter, # 50 µL filtered
    HTF     # 1000 µL filtered 
)

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
# await lh.stop()
await lh.setup(skip_autoload=True)

In [4]:
from pylabrobot.resources.plate import Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)
def VWR_96_wellplate_100_Vb(name: str, with_lid: bool = False) -> Plate:
  """
This plate is a VWR PCR plate 96 well low-profile, half-skirted, ABI-FAST type plate.
VWR cat no. 89218-296
It is half-skirted so it must reside in another plate like a Cor_96_wellplate_360ul_Fb
  """
  
  return Plate(
    name=name,
    size_x=127.76,
    size_y=85.48,
    size_z=20.0,
    # lid=lid,
    model=VWR_96_wellplate_100_Vb.__name__,
    ordered_items=create_ordered_items_2d(
      Well,
      num_items_x=12,
      num_items_y=8,
      dx=10.25,  # keeping costar measurement
      dy=10.5,  # 7.77 keeping costar measurement
      dz=8.5, # how high is well above base
      item_dx=9.0,
      item_dy=9.0,
      size_x=5.4,  # measured
      size_y=5.4,  # measured
      size_z=16.3, # measured well depth, costar + VWR plate height
      material_z_thickness=0.5,
      bottom_type=WellBottomType.V,
      cross_section_type=CrossSectionType.CIRCLE,
      max_volume=100,
    ),
  )

from typing import Optional

from pylabrobot.resources.height_volume_functions import (
  compute_height_from_volume_rectangle,
  compute_volume_from_height_rectangle,
)
from pylabrobot.resources.plate import Lid, Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)

In [5]:
###############################################################################
# 1) carriers, modules & labware
###############################################################################
# --- tip carrier -------------------------------------------------------------
# --- tip carrier -------------------------------------------------------------
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)
tiprack_1000 = HTF("tips_00")              # 1000 µL filter tips (slot-0)
tiprack_50   = TIP_50ul_w_filter("tips_01") #  50 µL filter tips (slot-1)
# mount the racks
tip_car[0] = tiprack_1000          # OR:  tip_car[0].assign_child_resource(tiprack_1000)
tip_car[1] = tiprack_50



# STANDARDS RACK
dwp_mod_dest   = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dest")
car_19 = MFX_CAR_L5_base(
    "car_19",
    modules={
        0: dwp_mod_dest
    }
)
lh.deck.assign_child_resource(car_19, rails=19)

# labware
tuberack_dest = opentrons_24_tuberack_generic_1point5ml_snapcap_short("dest_rack")
dest_offset_x = (127.76 - tuberack_dest._size_x) / 2
dest_offset_y = (85.48  - tuberack_dest._size_y) / 2

adapter_dest = TubeRackAdapter(
    name="dest_rack_adapter",
    size_x=127.76,
    size_y=85.48,
    size_z=tuberack_dest._size_z,              # external height of the frame
    model="tube_rack_adapter",
    dx=dest_offset_x,
    dy=dest_offset_y,
    dz=0,
    adapter_hole_size_x=tuberack_dest._size_x,
    adapter_hole_size_y=tuberack_dest._size_y,
    adapter_hole_size_z=tuberack_dest._size_z
)
adapter_dest.assign_child_resource(tuberack_dest)
dwp_mod_dest.assign_child_resource(adapter_dest)

# ----------carrier @ rail 13: water trough-------------------
# 96W, 100ul VWR PCR plate
dwp_mod_PCR = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_PCR")
dwp_mod_trough = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_trough")
car_13 = MFX_CAR_L5_base(
    "car_13",
    modules={
        0: dwp_mod_PCR,
        1: dwp_mod_trough,
    }
)
lh.deck.assign_child_resource(car_13, rails=13)
qPCR_plate = VWR_96_wellplate_100_Vb("qPCR_plate")
dwp_mod_PCR.assign_child_resource(qPCR_plate)
trough = AGenBio_1_troughplate_100000uL_Fl("water_trough")
dwp_mod_trough.assign_child_resource(trough)

# --- carrier @ rail-7: OT-2 tube rack with dsDNA ----------------------------
dwp_mod_src = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_src")
car_07 = MFX_CAR_L5_base(
    "car_07",
    modules={
        0: dwp_mod_src,
    }
)
lh.deck.assign_child_resource(car_07, rails=7)

tuberack_src = opentrons_24_tuberack_generic_1point5ml_snapcap_short("src_rack")
src_offset_x = (127.76 - tuberack_src._size_x) / 2
src_offset_y = (85.48  - tuberack_src._size_y) / 2

adapter_src = TubeRackAdapter(
    name="src_rack_adapter",
    size_x=127.76,
    size_y=85.48,
    size_z=tuberack_src._size_z,              # external height of the frame
    model="tube_rack_adapter",
    dx=src_offset_x,
    dy=src_offset_y,
    dz=0,
    adapter_hole_size_x=tuberack_src._size_x,
    adapter_hole_size_y=tuberack_src._size_y,
    adapter_hole_size_z=tuberack_src._size_z
)
adapter_src.assign_child_resource(tuberack_src)
dwp_mod_src.assign_child_resource(adapter_src)

In [13]:
###############################################################################
# 2) high-level parameters  ── adjust here if anything changes later
###############################################################################
CHANNEL_MM   = 6            # single channel we’ll use for the whole run
TIPRACK_50   = tiprack_50   # 50 µL filter tips (slot-1 on tip_car)

SRC_MM_TUBE  = tuberack_src["D6"]   # mastermix (carrier @ rail 7, tube rack “src_rack”)
DEST_PLATE   = qPCR_plate           # 96-well PCR plate (carrier @ rail 13)
START_TIP = "A1"
STANDARDS_RACK = tuberack_dest      # dilution-series rack (carrier @ rail 19)
STD_TUBES = {
    "dil1" : STANDARDS_RACK["A1"],  # 1st dilution
    "dil2" : STANDARDS_RACK["A2"],  # 2nd dilution
    "dil9" : STANDARDS_RACK["B3"],  # 9th dilution
    "dil15": STANDARDS_RACK["C3"],  # 15th dilution
}

TROUGH_WATER = trough["A1"]         # water control (carrier @ rail 13)

###############################################################################
# 3) helper utilities
###############################################################################
def plate_coords(rows, cols):
    """Return well objects for the supplied rows (string of A-H)
       and cols (iterable of ints 1-12)."""
    return [DEST_PLATE[f"{row}{col}"] for row in rows for col in cols]

ROW_LET = "ABCDEFGH"
def tip_sequence(start: str):
    sr, sc = start[0].upper(), int(start[1:])
    for r in ROW_LET[ROW_LET.index(sr):]:
        for c in range(sc if r == sr else 1, 13):
            yield f"{r}{c}"

tip_iter = tip_sequence(START_TIP)
tip_rack = TIPRACK_50

async def pick_up_50ul_tip(channel):
    tip_label = next(tip_iter)               # "E8", "E9", …
    tip_spot = tip_rack[tip_label]           # convert to TipSpot
    await lh.pick_up_tips(tip_spot, use_channels=[channel])

async def drop_tip(channel):
    await lh.drop_tips(use_channels=[channel])

###############################################################################
# 4) distribute mastermix (18 µL into every well)
###############################################################################
async def dispense_mastermix():
    rows = "ABCDEFGH"
    cols = range(1, 13)          # 1-12
    all_wells = plate_coords(rows, cols)

    # iterate two wells at a time (40 µL = 18+18+4 µL blow-out)
    well_pairs = [all_wells[i:i+2] for i in range(0, len(all_wells), 2)]

    # await pick_up_50ul_tip(CHANNEL_MM)

    # pre-wet tip for more accurate dispenses esp in first well
    # await lh.aspirate(
    #             SRC_MM_TUBE, 
    #             pre_wetting_volume=[40],
    #             vols=[40],
    #             use_channels=[CHANNEL_MM],
    #             lld_mode=[STARBackend.LLDMode.GAMMA]
    #             )
    # await lh.dispense(
    #             SRC_MM_TUBE,
    #             vols=[40],
    #             lld_mode=[STARBackend.LLDMode.GAMMA],
    #             use_channels=[CHANNEL_MM]
    #         )
    
    for pair in well_pairs:
        # 1. aspirate 40 µL mastermix
        try: #when liquid height gets too low
            await lh.aspirate(
                SRC_MM_TUBE, vols=[40],
                use_channels=[CHANNEL_MM],
                lld_mode=[STARBackend.LLDMode.GAMMA],
                immersion_depth=[3]
            )
        except: 
            await lh.aspirate(
                SRC_MM_TUBE, vols=[40], liquid_height=[2],
                use_channels=[CHANNEL_MM]
            )

        # 2. dispense 18 µL into each target well
        for dest in pair:
            await lh.dispense(
                dest, vols=[18],
                liquid_height=[2],
                use_channels=[CHANNEL_MM]
            )

        # 3. blow-out remaining ~4 µL back to mastermix tube
        try: #when liquid height gets too low
            await lh.dispense(
                SRC_MM_TUBE, vols=[4],   # 0 µL triggers blow-out in PLR
                use_channels=[CHANNEL_MM],
                lld_mode=[STARBackend.LLDMode.GAMMA],
                immersion_depth=[2],
                blow_out=[1]
            )
        except:
            await lh.dispense(
                SRC_MM_TUBE, vols=[4],   # 0 µL triggers blow-out in PLR
                use_channels=[CHANNEL_MM],
                liquid_height=[2],
                blow_out=[1]
            )

    await lh.discard_tips()

###############################################################################
# 5) helper to add 2 µL of a standard/water into six destination wells
###############################################################################

async def add_six_wells(src_tube, dest_wells):
    # await pick_up_50ul_tip(CHANNEL_MM, TIPRACK_50.next_tip())  # auto find next tip
    await pick_up_50ul_tip(CHANNEL_MM)  # auto find next tip
    await lh.aspirate(
        src_tube, 
        mix_volume = [50], #pre-moisten tip
        mix_cycles = [1], 
        vols=[24], # 20 ul so no gaps or stalls
        use_channels=[CHANNEL_MM],
        lld_mode=[STARBackend.LLDMode.GAMMA], # vol = 900ul, no chance of bottomming out. 
        immersion_depth=[2]
    )
    # a quick dispense back into the tube to prime the pump
    await lh.dispense( 
        src_tube, 
        vols=[4], # 20 ul so no gaps or stalls
        use_channels=[CHANNEL_MM],
        lld_mode=[STARBackend.LLDMode.GAMMA], # vol = 900ul, no chance of bottomming out. 
        immersion_depth=[2]
    )
    for dest in dest_wells:
        await lh.dispense(
            dest, vols=[2],
            use_channels=[CHANNEL_MM],
            liquid_height=[2],
            settling_time=[2]# should make contact with 18ul about halfway down
        )

    await lh.discard_tips()

###############################################################################
# 6) distribute all 15 standards (6 × replicates) + water controls
###############################################################################
async def dispense_standards():
    """
    Adds 2 µL of each of the 15 serial-dilution tubes (A1-C3, row-major)
    into six wells of the qPCR plate, then fills H7-H12 with water.
    ─────────────────────────────────────────────────────────────────────────
    Tube-rack layout (24-well, 6 cols × 4 rows):
        A1-A6, B1-B6, C1-C3  ← 15 tubes
    Plate layout (96-well):
        • Each standard gets 6 technical replicates (90 wells total)
        • Remaining 6 wells (H7-H12) receive water controls
    """

    # mapping: tube position → destination wells (list of six qPCR-plate wells)
    dest_map = {
        # first 6 standards → A–F rows, columns 1-6
        "A1": plate_coords("A", range(1,  7)),   # A1-A6
        "A2": plate_coords("B", range(1,  7)),   # B1-B6
        "A3": plate_coords("C", range(1,  7)),   # C1-C6
        "A4": plate_coords("D", range(1,  7)),   # D1-D6
        "A5": plate_coords("E", range(1,  7)),   # E1-E6
        "A6": plate_coords("F", range(1,  7)),   # F1-F6

        # next 6 standards → A–F rows, columns 7-12
        "B1": plate_coords("G", range(1, 7)),   # A7-A12
        "B2": plate_coords("H", range(1, 7)),   # B7-B12
        "B3": plate_coords("A", range(7, 13)),   # C7-C12
        "B4": plate_coords("B", range(7, 13)),   # D7-D12
        "B5": plate_coords("C", range(7, 13)),   # E7-E12
        "B6": plate_coords("D", range(7, 13)),   # F7-F12

        # last 3 standards → G/H rows
        "C1": plate_coords("E", range(7,  13)),   # G1-G6
        "C2": plate_coords("F", range(7,  13)),   # H1-H6
        "C3": plate_coords("G", range(7, 13)),   # G7-G12
    }

    # loop through every standard tube and dispense 6 × 2 µL
    for tube_pos, dest_wells in dest_map.items():
        src_tube = STANDARDS_RACK[tube_pos]
        # await add_six_wells(src_tube, dest_wells)
        await add_six_wells(STANDARDS_RACK["A1"], dest_wells) #for testing purposes

    # finally, water controls into H7-H12
    water_wells = plate_coords("H", range(7, 13))   # H7-H12
    await add_six_wells(TROUGH_WATER, water_wells)

###############################################################################
# 7) main entry-point
###############################################################################
async def run_protocol():
    # await lh.home()                # optional – start from home
    # await dispense_mastermix()
    
    await dispense_standards()
    print ("SuccessfullyCompleted")# await lh.home()                # park at home when finished

###############################################################################
# 8) kick off
###############################################################################

await run_protocol()

SuccessfullyCompleted


In [ ]:
# await lh.dispense(trough["A1"], vols=[900], liquid_height=[2], use_channels=[CHANNEL_WATER])
# await lh.dispense(SRC_MM_TUBE, vols=[2], liquid_height=[2], use_channels=[CHANNEL_MM])
# await lh.drop_tips(tiprack_1000["A1"], use_channels=[2])
# await lh.discard_tips()
# await lh.stop()